# Environment

> **Nota:** Este notebook requer que `hate-br-lex.csv` e `ited-br-lex.csv` contenham as colunas `feat_pos_tokens`, `feat_neg_tokens` e `feat_nto_tokens` (listas de tokens). Caso essas colunas não existam, execute o notebook `3a_feat_lex.ipynb` com a versão completa da função `extract_lex_features` que inclui os tokens.

In [ ]:
import os
os.chdir("../..")

In [ ]:
import ast

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import spacy

from src.utils.features import load_features
from src.utils.path import PROCESSED_DATA_PATH, PROCESSED_LEXICONS_PATH, FEATURES_PATH

In [ ]:
nlp = spacy.load("pt_core_news_lg")

## Load Data

### Lexicons

In [ ]:
liwc = pd.read_csv(f"{PROCESSED_LEXICONS_PATH}/liwc.csv")
sentilex_df = pd.read_csv(f"{PROCESSED_LEXICONS_PATH}/sentilex.csv")
wordnetaffect = pd.read_csv(f"{PROCESSED_LEXICONS_PATH}/wordnetaffect.csv")

In [ ]:
lex = (
    pd.concat([liwc, sentilex_df, wordnetaffect])
    .sort_values("polarity")
    .drop_duplicates(subset=["word"], keep="first")
    .reset_index(drop=True)
)

In [ ]:
pos_set = set(lex.loc[lex.polarity == 1, "word"].str.lower())
neg_set = set(lex.loc[lex.polarity == -1, "word"].str.lower())
nto_set = set(lex.loc[lex.polarity == 0, "word"].str.lower())

### Copora with Lex Features

In [ ]:
hate = load_features(
    base_path=f"{PROCESSED_DATA_PATH}/hate-br.csv",
    feature_paths=[f"{FEATURES_PATH}/hate-br-lex.csv"]
)

ited = load_features(
    base_path=f"{PROCESSED_DATA_PATH}/ited-br.csv",
    feature_paths=[f"{FEATURES_PATH}/ited-br-lex.csv"]
)

## Token Extraction

In [ ]:
nlp_extract = spacy.load("pt_core_news_lg", disable=["tagger", "parser", "ner", "lemmatizer"])

In [ ]:
def extract_tokens(text):
    doc = nlp_extract(str(text))
    toks = [t.text.lower() for t in doc if not t.is_stop]
    return {
        "feat_pos_tokens": str([t for t in toks if t in pos_set]),
        "feat_neg_tokens": str([t for t in toks if t in neg_set]),
        "feat_nto_tokens": str([t for t in toks if t in nto_set]),
    }

for df in [hate, ited]:
    tokens = df["text"].apply(extract_tokens).apply(pd.Series)
    df[tokens.columns] = tokens

# POS Tagging

In [ ]:
def tag_keywords(row, column):
    text = row["text"]
    try:
        keywords = ast.literal_eval(row[column])
    except (ValueError, TypeError):
        return {}
    if not keywords:
        return {}
    doc = nlp(text)
    return {token.text: token.pos_ for token in doc if token.text.lower() in keywords}

def tag_keywords_dataframe(df_input, token_columns):
    df = df_input.copy()
    for col_name in token_columns:
        df[f"{col_name}_pos"] = df.apply(lambda row: tag_keywords(row, col_name), axis=1)
    return df

In [ ]:
token_columns = ["feat_pos_tokens", "feat_neg_tokens", "feat_nto_tokens"]

hate = tag_keywords_dataframe(hate, token_columns)
ited = tag_keywords_dataframe(ited, token_columns)

pos_feat_cols = [f"{col}_pos" for col in token_columns]
hate[pos_feat_cols].to_csv(f"{FEATURES_PATH}/hate-br-pos-tags.csv", index=False)
ited[pos_feat_cols].to_csv(f"{FEATURES_PATH}/ited-br-pos-tags.csv", index=False)

# Frequency Analysis

In [ ]:
def calculate_frequency_pos(df, pos_column):
    pos_tags_list = df[pos_column].apply(lambda d: list(d.values()) if isinstance(d, dict) else [])
    exploded_tags = pos_tags_list.explode().dropna()
    count = exploded_tags.value_counts()
    percentage = exploded_tags.value_counts(normalize=True) * 100
    tag = pos_column.split("_")[1]
    return pd.DataFrame({
        "pos_tag": count.index,
        f"abs_freq_{tag}": count.values,
        f"rel_freq_{tag}": percentage.values.round(2)
    })

In [ ]:
freq_pos = calculate_frequency_pos(hate, "feat_pos_tokens_pos")
freq_neg = calculate_frequency_pos(hate, "feat_neg_tokens_pos")
freq_nto = calculate_frequency_pos(hate, "feat_nto_tokens_pos")

In [ ]:
df_pos_tag = freq_pos.merge(freq_neg, on="pos_tag", how="outer").merge(freq_nto, on="pos_tag", how="outer").fillna(0)

In [ ]:
df_pos_tag.to_csv(f"{FEATURES_PATH}/hate-br-pos-freq.csv", index=False)

# Analysis

In [ ]:
selected_tags = ["ADJ", "ADV", "NOUN", "PROPN", "VERB"]
polarities = ["Positivo", "Negativo", "Neutro"]
cols = ["rel_freq_pos", "rel_freq_neg", "rel_freq_nto"]

df_selected = df_pos_tag[df_pos_tag["pos_tag"].isin(selected_tags)]

df_melted = pd.melt(df_selected, id_vars="pos_tag", value_vars=cols, var_name="sentiment", value_name="relative_frequency")
df_melted["sentiment"] = df_melted["sentiment"].map({"rel_freq_pos": "Positivo", "rel_freq_neg": "Negativo", "rel_freq_nto": "Neutro"})

plt.figure(figsize=(8, 5))
for tag in selected_tags:
    subset = df_melted[df_melted["pos_tag"] == tag]
    plt.plot(subset["sentiment"], subset["relative_frequency"], marker="o", label=tag)
plt.title("Frequência Relativa por Classe e Polaridade")
plt.xlabel("Polaridade")
plt.ylabel("Frequência Relativa (%)")
plt.legend(title="Classe gramatical")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df_selected.set_index("pos_tag")[cols].T, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title("Heatmap de Frequência Relativa por Polaridade e Classe Gramatical")
plt.xlabel("Classe gramatical")
plt.ylabel("Polaridade")
plt.yticks(ticks=np.arange(len(polarities)), labels=polarities, rotation=0)
plt.tight_layout()
plt.show()